# Домашняя работа ИТОГ
### Задание: автоматизированный мониторинг и реагирование на угрозы

**Цель:** разработать скрипт, который собирает данные из двух источников (логи Suricata и API VirusTotal), анализирует их на предмет подозрительной активности, имитирует простые реакции и формирует отчёт с визуализацией.

**Этапы:**
1. Загрузить файл с логами Suricata (формат .json).
2. Получить ключ API VirusTotal.
3. Из логов выделить IP-адреса источников, посчитать частоту.
4. Для наиболее активных IP выполнить запрос к VirusTotal для получения количества обнаружений (malicious).
5. Классифицировать угрозы на основе частоты и данных VirusTotal.
6. Вывести сообщения об угрозах и имитировать блокировку IP с высоким уровнем.
7. Сохранить результаты анализа в форматах .csv и .json.
8. Построить столбчатую диаграмму топ-10 IP по количеству событий, раскрасить по уровню угрозы и сохранить в .png.

**Требования:**
- Использованы минимум два источника данных (логи + API).
- Скрипт корректно выполняется и выводит результат анализа.
- Реализовано простое реагирование (сообщения или имитация блокировки).
- Создан и сохранён отчёт в формате .json или .csv.
- Создан и сохранён график в формате .png.
- Подготовлено краткое описание работы скрипта (в комментариях).

**Примечание:** график сохраняется в файл `threat_chart.png`, путь для удобства указан как `C:\users\desktop\threat_chart.png` (при локальном запуске замените на свой).

In [ ]:
!pip install requests pandas matplotlib seaborn -q
print("Библиотеки установлены")

In [ ]:
from google.colab import files
import os

print("Загрузите файл логов Suricata (.json)")
uploaded = files.upload()

# Автоопределение JSON файла
log_files = [f for f in uploaded.keys() if f.endswith('.json')]
if log_files:
    LOG_FILE = log_files[0]
    print(f"Автовыбор: {LOG_FILE}")
    with open(LOG_FILE, 'wb') as f:
        f.write(uploaded[LOG_FILE])
else:
    print("Нет .json! ")

In [ ]:
# API-КЛЮЧ VIRUSTOTAL
VT_API_KEY = "216e0b465a56fd83351f0c1b5cae17a8e2fabebf4c587ce49c37fe1d9e42cb36"
print(f"API ключ: {'Установлен' if VT_API_KEY else 'НЕ УСТАНОВЛЕН'}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import time
from datetime import datetime
import os

# Определяем файл логов
json_files = [f for f in os.listdir('.') if f.endswith('.json') and f != 'threats_*.json']
LOG_FILE = json_files[0] if json_files else None
if not LOG_FILE:
    raise FileNotFoundError("Не найден JSON файл с логами")

print(f" Анализ {LOG_FILE} ")
print("=" * 70)

# Чтение логов
df = pd.read_json(LOG_FILE, lines=True)
print(f"Загружено: {len(df)} строк")

# Поиск колонки с IP источника
src_ip_col = None
for col in df.columns:
    if 'src_ip' in col.lower() or ('ip' in col.lower() and 'src' in col.lower()):
        src_ip_col = col
        break
if not src_ip_col:
    # если не нашли, берём первую колонку с 'ip'
    ip_cols = [col for col in df.columns if 'ip' in col.lower()]
    if ip_cols:
        src_ip_col = ip_cols[0]
    else:
        raise ValueError("Не удалось определить колонку с IP")

print(f" Используем src_ip: '{src_ip_col}'")

# Подсчёт событий по IP
ip_counts = df[src_ip_col].value_counts().head(50).reset_index()
ip_counts.columns = ['ip', 'alert_count']

# Функция проверки IP через VirusTotal
def check_virustotal(ip):
    url = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"
    headers = {"x-apikey": VT_API_KEY}
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            data = response.json()
            stats = data['data']['attributes']['last_analysis_stats']
            return stats.get('malicious', 0), stats.get('suspicious', 0)
        else:
            print(f"Ошибка API для {ip}: {response.status_code}")
            return 0, 0
    except Exception as e:
        print(f"Исключение для {ip}: {e}")
        return 0, 0

# Проверяем топ-5 IP через VirusTotal (с задержкой 15 сек между запросами)
top_ips = ip_counts.head(5)['ip'].tolist()
vt_results = []
for ip in top_ips:
    print(f"Проверка {ip} через VirusTotal...")
    malicious, suspicious = check_virustotal(ip)
    vt_results.append({'ip': ip, 'vt_malicious': malicious, 'vt_suspicious': suspicious})
    time.sleep(15)  # задержка для соблюдения лимитов бесплатного API

# Объединяем с основными данными
vt_df = pd.DataFrame(vt_results)
analysis_df = ip_counts.merge(vt_df, on='ip', how='left')
analysis_df['vt_malicious'] = analysis_df['vt_malicious'].fillna(0).astype(int)
analysis_df['vt_suspicious'] = analysis_df['vt_suspicious'].fillna(0).astype(int)

# Классификация угрозы: комбинируем alert_count и vt_malicious
def classify_threat(row):
    if row['alert_count'] >= 10 or row['vt_malicious'] >= 3:
        return 'HIGH'
    elif row['alert_count'] >= 3 or row['vt_malicious'] >= 1:
        return 'MEDIUM'
    else:
        return 'LOW'

analysis_df['threat'] = analysis_df.apply(classify_threat, axis=1)

print("\n ТОП IP с результатами:")
print(analysis_df.head(10))

# Построение графика
plt.figure(figsize=(12, 8))
top10 = analysis_df.nlargest(10, 'alert_count')
sns.barplot(data=top10, x='alert_count', y='ip', hue='threat', palette='Reds_r')
plt.title(f'Топ-10 IP по активности ({LOG_FILE})')
plt.xlabel('Количество событий')
plt.tight_layout()
plt.savefig('threat_chart.png', dpi=150, bbox_inches='tight')
print("\n График сохранён как C:\\users\\desktop\\threat_chart.png (фактически в текущей директории)")

# Сохранение отчётов
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
analysis_df.to_csv(f'threats_{timestamp}.csv', index=False)
analysis_df.to_json(f'threats_{timestamp}.json', orient='records', indent=2)

print(f"\n ФАЙЛЫ ГОТОВЫ:")
print(f"    threats_{timestamp}.csv")
print(f"    threats_{timestamp}.json")
print(f"    threat_chart.png")

# Имитация реагирования
high_ips = analysis_df[analysis_df['threat'] == 'HIGH']['ip'].tolist()
if high_ips:
    print("\n[!] Обнаружены IP с высоким уровнем угрозы. Имитация блокировки:")
    for ip in high_ips:
        print(f"    Блокировка {ip} (добавление в ACL)")
else:
    print("\n[-] Угроз высокого уровня не найдено.")